# Continuous Beam Verification — `ContinuousBeamAnalysis`

Verifies `ContinuousBeamAnalysis` from `scite/beam/continuous/continuous_beam.py`
against analytical references and checks internal consistency.

**Algorithm:** Each span is treated as an independent SSB.  The unknown hogging
moment $M_\mathrm{hog}$ at the interior support is found by enforcing
rotation compatibility:
$$
\varphi_a(L_a;\, M_\mathrm{hog}) = \varphi_b(0;\, M_\mathrm{hog})
$$
End-rotations are integrated numerically from the $M$–$\kappa$ curve with the
SSB displacement BC $w(0)=w(L)=0$.

**Sign convention (sagging positive)**  
- $M>0$ → sagging (tension at bottom).  
- $M_\mathrm{hog}\ge0$ is the *magnitude*; it enters the moment expressions as
  $-M_\mathrm{hog}$.  
- $\varphi = -dw/dx$ with $w$ downward positive.

**Scope**
- Section 1 — Imports and reference floor system (M–κ with κ_yield and M_cr)
- Section 2 — Elastic verification: three-moment equation vs. classical formula
- Section 3 — Displacement boundary conditions $w(0)=w(L)=0$
- Section 4 — Residuum sign check (bracketing validity)
- Section 5 — Elastic limit: `ContinuousBeamAnalysis` on an uncracked section
- Section 6 — Equal-span nonlinear solve (cracking redistribution)
- Section 6b — **Unequal spans (S = 1.3): plastic hinge and sufficient rotational capacity**
- Section 7 — Unequal spans: span-ratio sweep
- Section 8 — Full plotting suite: scheme, M, w, redistribution


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from scite.beam.continuous import ContinuousBeamAnalysis, _phi_ssb
from scite.beam.bending.beam_deflection import BeamDeflectionAnalysis

# ── cross-section design ──────────────────────────────────────────────────────
from scite.cs_design.shapes import RectangularShape
from scite.cs_design.cross_section import CrossSection
from scite.cs_design.reinforcement import ReinforcementLayer, ReinforcementLayout
from scite.matmod.ec2_parabola_rectangle import EC2ParabolaRectangle
from scite.matmod.steel_reinforcement import create_steel

%matplotlib inline
plt.rcParams.update({'figure.dpi': 130, 'font.size': 10})
BLUE, RED, GREEN = '#1f77b4', '#d62728', '#2ca02c'
print('Imports OK')

---
## 1  Reference RC beam — cross-section and M-κ capacity

The reference section is a **symmetric doubly-reinforced rectangular RC beam**
(200 × 400 mm, C30/37, B500B):

| Location | Reinforcement | Area | z from bottom |
|----------|--------------|------|--------------|
| **Bottom** (sagging tension at midspan) | 2ø16 | 402 mm² | 45 mm |
| **Top** (hogging tension at interior support) | 2ø16 | 402 mm² | 355 mm |

Equal areas at top and bottom → **symmetric section** → the M-κ curve is
anti-symmetric about the origin: $M(-\kappa) = -M(\kappa)$.

`BeamDeflectionAnalysis` exploits this via its *symmetric-section approximation*
in `get_kappa_x_from_M`: one M-κ pass (positive κ, sagging) is computed, and the
hogging branch is obtained by reflection.  For an *asymmetric* section (different
top/bottom steel) two separate M-κ analyses would be required.


In [ ]:
# ── Reference rectangular RC cross-section (200×400 mm, 2ø16 B500B each face) ──
b_ref, h_ref = 200.0, 400.0   # mm
A_s_ref      = 402.0           # mm²  (2ø16 per face)
z_s_ref      = 45.0            # mm  from bottom  (cover ≈ 30 mm + ø/2 = 8 mm)
z_s_top      = h_ref - z_s_ref # mm  from bottom  (= 355 mm)
L_ref        = 6000.0          # mm  span (= 6 m)

concrete_ref = EC2ParabolaRectangle(f_ck=30.0, alpha_cc=0.85, gamma_c=1.5)
steel        = create_steel('B500B')

shape_rect = RectangularShape(b=b_ref, h=h_ref)
layer_bot  = ReinforcementLayer(material=steel, A_s=A_s_ref, z=z_s_ref)
layer_top  = ReinforcementLayer(material=steel, A_s=A_s_ref, z=z_s_top)
reinf_ref  = ReinforcementLayout(layers=[layer_bot, layer_top])
cs_rect    = CrossSection(shape=shape_rect, concrete=concrete_ref,
                          reinforcement=reinf_ref)

bda = BeamDeflectionAnalysis(
    cs=cs_rect, L=L_ref, load_type='dist',
    n_x=200, n_kappa=150, n_load_steps=41,
)

# Quasi-permanent SLS design load: self-weight + dead load + ψ₂ × variable load.
# Notation follows the floor-package convention: p_Ed_qp (SLS) vs. p_Ed_u (ULS).
p_Ed_qp = 0.9 * bda.F_R

print(f'Cross-section: b={b_ref:.0f} mm, h={h_ref:.0f} mm')
print(f'  Bottom:  2ø16  A_s,bot = {A_s_ref:.0f} mm²  at z = {z_s_ref:.0f} mm from bottom')
print(f'  Top   :  2ø16  A_s,top = {A_s_ref:.0f} mm²  at z = {z_s_top:.0f} mm from bottom')
print(f'M_R      = {bda.M_R:.2f} kN·m   (moment capacity, symmetric section)')
print(f'p_Ed_u   = {bda.F_R:.4f} N/mm  = {bda.F_R:.3f} kN/m  (ULS design load, p_Ed_u)')
print(f'p_Ed_qp  = {p_Ed_qp:.4f} N/mm  = {p_Ed_qp:.3f} kN/m  (quasi-permanent SLS load, p_Ed_qp)')
print(f'Span     = {bda.L:.0f} mm')


In [ ]:
from matplotlib.patches import Rectangle, Circle

fig, (ax_cs, ax_mk) = plt.subplots(1, 2, figsize=(13, 6),
                                    gridspec_kw={'width_ratios': [1, 1.6]})

# ── 1.  Cross-section sketch ─────────────────────────────────────────────────
b, h, r = b_ref, h_ref, 8.0   # bar drawn radius ≈ ø16 scale
n = 2                           # bars per face

# Concrete body
ax_cs.add_patch(Rectangle((0, 0), b, h, facecolor='#d4d4d4', edgecolor='k', lw=2))
# Stirrup (simplified)
sc = 32.0
ax_cs.add_patch(Rectangle((sc, sc), b - 2*sc, h - 2*sc,
    facecolor='none', edgecolor='#555', lw=1, ls='--', alpha=0.7))

# Bottom bars (sagging tension at midspan)
for i in range(n):
    xc = b * (i + 1) / (n + 1)
    ax_cs.add_patch(Circle((xc, z_s_ref), r, fc='#cc2222', ec='k', lw=0.8, zorder=5))

# Top bars (hogging tension at interior support)
for i in range(n):
    xc = b * (i + 1) / (n + 1)
    ax_cs.add_patch(Circle((xc, z_s_top), r, fc='#cc2222', ec='k', lw=0.8, zorder=5))

# Dimension annotations
m = 30.0
ax_cs.annotate('', xy=(0, -m), xytext=(b, -m),
    arrowprops=dict(arrowstyle='<->', color='k', lw=1.2))
ax_cs.text(b/2, -m - 12, f'b = {b:.0f} mm', ha='center', va='top', fontsize=10)

ax_cs.annotate('', xy=(-m, 0), xytext=(-m, h),
    arrowprops=dict(arrowstyle='<->', color='k', lw=1.2))
ax_cs.text(-m - 8, h/2, f'h = {h:.0f} mm',
    ha='right', va='center', fontsize=10, rotation=90)

ax_cs.text(b/2, z_s_ref - 24,
    f'2ø16  A_s = {A_s_ref:.0f} mm²  (z = {z_s_ref:.0f} mm)',
    ha='center', va='top', fontsize=9, color='#aa1111')
ax_cs.text(b/2, z_s_top + 22,
    f'2ø16  A_s = {A_s_ref:.0f} mm²  (z = {z_s_top:.0f} mm)',
    ha='center', va='bottom', fontsize=9, color='#aa1111')

ax_cs.set_xlim(-70, b + 30)
ax_cs.set_ylim(-65, h + 35)
ax_cs.set_aspect('equal')
ax_cs.axis('off')
ax_cs.set_title(f'Cross-section  {b:.0f} × {h:.0f} mm  (C30/37, B500B)',
    fontsize=11, fontweight='bold')

# ── 2.  Full M-κ curve (positive pass + anti-symmetric mirror) ───────────────
mk         = bda.mk                     # MKappaAnalysis, already solved
kappa_pos  = mk.kappa_values * 1e3      # [1/mm] → [1/m]
M_pos      = mk.M_values                # [kN·m]
M_peak     = float(M_pos.max())

# Symmetric section → hogging branch is the sign-mirror of the sagging branch
kappa_full = np.concatenate([-kappa_pos[::-1], kappa_pos])
M_full     = np.concatenate([-M_pos[::-1],      M_pos])

# ── Yield curvature and plastic rotation capacity ────────────────────────────
# κ_yield: analytical estimate — bottom steel yields at ε_s = ε_yd = f_yd / E_s
d_ref    = h_ref - z_s_ref                              # effective depth [mm]
f_yd     = 500.0 / 1.15                                 # design yield strength [MPa]
eps_yd   = f_yd / 200_000.0                             # yield strain [-]
x_yield  = (A_s_ref * f_yd                              # neutral axis depth at yield [mm]
            / (0.8 * concrete_ref.f_ck * b_ref * 0.8))  # (EC2 rect. block, λ=0.8)
kappa_y  = eps_yd / (d_ref - x_yield) * 1e3            # 1/mm → 1/m
kmax     = float(kappa_pos[-1])                         # max curvature in sweep

# Cracking moment (gross section, no reinforcement contribution)
f_ctm = 0.3 * float(concrete_ref.f_ck) ** (2.0 / 3.0)  # mean tensile strength [MPa]
I_g   = b_ref * h_ref ** 3 / 12.0                        # gross moment of inertia [mm⁴]
M_cr  = f_ctm * I_g / (h_ref / 2.0) / 1e6               # cracking moment [kN·m]

# ── Shade the plastic rotation zone (κ > κ_y → yield plateau utilised) ───────
ax_mk.axvspan( kappa_y,  kmax, alpha=0.13, color='gold', zorder=0,
    label=f'Plastic rotation zone  (κ > κ_y)')
ax_mk.axvspan(-kmax, -kappa_y, alpha=0.13, color='gold', zorder=0)
ax_mk.axvline( kappa_y, color='darkorange', ls=':', lw=1.3,
    label=f'κ_y = {kappa_y:.4f} 1/m  (bottom steel yields)')
ax_mk.axvline(-kappa_y, color='darkorange', ls=':', lw=1.3)

# Cracking moment reference lines
ax_mk.axhline( M_cr, color='#888', ls='-.', lw=1.0,
    label=f'M_cr ≈ {M_cr:.1f} kN·m  (gross section)')
ax_mk.axhline(-M_cr, color='#888', ls='-.', lw=1.0)

ax_mk.plot(kappa_full, M_full, color='#1f77b4', lw=2,
    label='M-κ  (sagging ↗ / hogging ↙, symmetric section)')
ax_mk.axhline( M_peak, color='green', ls='--', lw=1.2,
    label=f'+M_R = +{M_peak:.1f} kN·m')
ax_mk.axhline(-M_peak, color='green', ls='--', lw=1.2,
    label=f'−M_R = −{M_peak:.1f} kN·m')
ax_mk.axhline(0, color='k', lw=0.8)
ax_mk.axvline(0, color='k', lw=0.8)

ax_mk.set_xlabel('Curvature  κ  [1/m]', fontsize=11)
ax_mk.set_ylabel('Moment  M  [kN·m]', fontsize=11)
ax_mk.set_title('M-κ capacity — symmetric section (one pass + mirror)',
    fontsize=11, fontweight='bold')
ax_mk.legend(fontsize=9, loc='upper left')
ax_mk.grid(True, alpha=0.3, ls='--')

plt.tight_layout()
plt.show()

print(f'Symmetric section:  ±M_R = ±{M_peak:.2f} kN·m')
print(f'M_cr (gross section approx)   = {M_cr:.2f} kN·m')
print(f'κ_yield (bottom steel)        = {kappa_y:.5f} 1/m  '
      f'(x/d = {x_yield/d_ref:.3f}  ← EC2 limit: 0.45 class B, 0.35 class C)')
print(f'Curvature ductility  κ_u/κ_y  ≈ {kmax/kappa_y:.1f}  (wide plastic plateau)')
print(f'Note: equal top/bottom steel → M-κ is anti-symmetric.')
print(f'      BeamDeflectionAnalysis uses the positive (sagging) branch only')
print(f'      and applies sign(M) × κ(|M|) for hogging zones.')


---
## 2  Elastic verification — three-moment equation

For a two-span continuous beam with equal spans $L_a = L_b = L$ and uniform EI,
Clapeyron's three-moment equation gives:

$$
M_\mathrm{hog,el} = \frac{q\,(L_a^3 + L_b^3)}{8\,(L_a + L_b)}
  \xrightarrow{L_a=L_b} \frac{q\,L^2}{8}
$$

We verify the `M_hog_elastic` property against the closed-form for equal spans.

In [ ]:
cba = ContinuousBeamAnalysis(span_left=bda, span_right=bda, q=p_Ed_qp)

La = bda.L
Lb = bda.L
M_el_formula  = p_Ed_qp * La**2 / 8.0                       # equal-span shortcut
M_el_property = cba.M_hog_elastic                            # three-moment equation
M_el_general  = p_Ed_qp * (La**3 + Lb**3) / (8*(La+Lb))     # general form

print(f'M_hog,el (q L²/8)               = {M_el_formula  / 1e6:.4f} kN·m')
print(f'M_hog,el (three-moment, general) = {M_el_general  / 1e6:.4f} kN·m')
print(f'M_hog,el (property)              = {M_el_property / 1e6:.4f} kN·m')

assert abs(M_el_property - M_el_formula) / M_el_formula < 1e-10, \
    'M_hog_elastic property mismatch for equal spans'
print('Equal-span check PASSED  ✓')

# ── Unequal-span check: S = 2.0 ──────────────────────────────────────────────
S    = 2.0          # L_b / L_a
La2  = 5000.0       # mm
Lb2  = S * La2

bda_a = BeamDeflectionAnalysis(cs=cs_rect, L=La2, load_type='dist',
                                n_x=200, n_kappa=150, n_load_steps=41)
bda_b = BeamDeflectionAnalysis(cs=cs_rect, L=Lb2, load_type='dist',
                                n_x=200, n_kappa=150, n_load_steps=41)

cba2 = ContinuousBeamAnalysis(span_left=bda_a, span_right=bda_b, q=p_Ed_qp)
M_el_ref  = p_Ed_qp * (La2**3 + Lb2**3) / (8*(La2 + Lb2))
M_el_prop = cba2.M_hog_elastic
assert abs(M_el_prop - M_el_ref) / M_el_ref < 1e-10, \
    'M_hog_elastic property mismatch for unequal spans'
print(f'Unequal-span (S={S}) check PASSED  ✓')


---
## 3  Displacement boundary conditions $w(0)=w(L)=0$

The `_phi_ssb` helper integrates curvature with integration constant
$C_1 = -\langle\varphi_\mathrm{int}\rangle$ so that $w(L)=0$ is satisfied.
We verify this for both spans under the solved $M_\mathrm{hog}$.

In [ ]:
M_nl = cba.solve()
print(f'M_hog_solved = {M_nl/1e6:.4f} kN·m')

x_a, w_a, x_b, w_b = cba.w_profiles(M_nl)

tol = 1e-6   # mm — effectively zero relative to typical deflections
assert abs(w_a[ 0]) < tol, f'w_a(0) = {w_a[ 0]:.2e} mm — BC violated'
assert abs(w_a[-1]) < tol, f'w_a(L) = {w_a[-1]:.2e} mm — BC violated'
assert abs(w_b[ 0]) < tol, f'w_b(0) = {w_b[ 0]:.2e} mm — BC violated'
assert abs(w_b[-1]) < tol, f'w_b(L) = {w_b[-1]:.2e} mm — BC violated'

print(f'w_a(0)  = {w_a[ 0]:.2e} mm  ✓')
print(f'w_a(La) = {w_a[-1]:.2e} mm  ✓')
print(f'w_b(0)  = {w_b[ 0]:.2e} mm  ✓')
print(f'w_b(Lb) = {w_b[-1]:.2e} mm  ✓')
print(f'Displacement BCs satisfied  ✓')
print(f'Peak mid-span deflections:  left {max(w_a):.2f} mm,  right {max(w_b):.2f} mm')

---
## 4  Residuum sign check — bracketing validity

The brentq solver requires a sign change on the bracket
$[0,\; 1.5\,M_\mathrm{hog,el}]$.

- At $M_\mathrm{hog}=0$: no hogging at the support, left end-rotation
  $\varphi_a(L_a)>0$ (upward slope), right end-rotation $\varphi_b(0)<0$
  (downward slope) → residuum $R>0$.
- At $M_\mathrm{hog}\to\infty$: the hogging dominates, reversing both end
  rotations → residuum $R<0$.

In [ ]:
M_bracket_lo = 0.0
M_bracket_hi = 1.5 * cba.M_hog_elastic

R_lo = cba.residuum(M_bracket_lo + 1.0)   # avoid exact zero
R_hi = cba.residuum(M_bracket_hi)

print(f'R(0+)              = {R_lo:+.6f}')
print(f'R(1.5 * M_hog,el)  = {R_hi:+.6f}')

assert R_lo > 0, 'Expected R > 0 at lower bracket'
assert R_hi < 0, 'Expected R < 0 at upper bracket'
print('Sign change confirmed — bracket valid  ✓')

# Plot residuum curve
M_arr = np.linspace(0.1 * cba.M_hog_elastic, 1.5 * cba.M_hog_elastic, 80)
R_arr = np.array([cba.residuum(m) for m in M_arr])

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.plot(M_arr / 1e6, R_arr, color=BLUE, lw=1.5)
ax.axhline(0, color='k', lw=0.8, ls='--')
ax.axvline(M_nl / 1e6, color=RED, lw=1.0, ls=':', label=f'$M_{{hog}}$ = {M_nl/1e6:.2f} kN·m')
ax.axvline(cba.M_hog_elastic / 1e6, color=GREEN, lw=1.0, ls=':',
           label=f'$M_{{hog,el}}$ = {cba.M_hog_elastic/1e6:.2f} kN·m')
ax.set_xlabel('$M_{hog}$ [kN·m]')
ax.set_ylabel('Residuum $R$ [rad]')
ax.set_title('Rotation-compatibility residuum')
ax.legend()
ax.grid(True, lw=0.4)
plt.tight_layout()
plt.show()

---
## 5  Elastic limit — nonlinear solve on an uncracked section

When the section stays fully uncracked (linear $M$–$\kappa$), the nonlinear
solver must recover the same $M_\mathrm{hog}$ as the three-moment equation.
We test this by using only the **first segment** of the $M$–$\kappa$ table
(effectively linear) via a very small load $q_\mathrm{tiny}$.

In [ ]:
# The MKappaAnalysis κ-sweep starts at kappa_min = kappa_max * 0.001 (not κ=0),
# so M_values[0] > 0.  Any load that puts M_max *below* M_values[0] causes
# np.interp to return the constant kappa_ref[0] everywhere — piecewise-constant
# curvature — and the solver gives M_hog ≠ qL²/8.
#
# Fix: use a q that places M_max firmly *inside* the first (linear) segment of
# the M-κ table, i.e.  M_values[0] < M_max < M_values[1].

idx_peak   = int(np.argmax(bda.mk.M_values))
M_table_0  = float(bda.mk.M_values[0]) * 1e6   # N·mm  — first table entry
M_table_1  = float(bda.mk.M_values[1]) * 1e6   # N·mm  — second table entry

# Choose M_max = 3 × M_table_0 → well above table start, below M_table_1
q_elastic    = 3.0 * M_table_0 * 8.0 / bda.L ** 2
M_max_at_el  = q_elastic * bda.L ** 2 / 8.0

print(f'M-κ table start  = {M_table_0 / 1e6:.4f} kN·m   (= M_values[0])')
print(f'M-κ table 2nd pt = {M_table_1 / 1e6:.4f} kN·m   (= M_values[1])')
print(f'q_elastic        = {q_elastic:.4e} N/mm')
print(f'M_max            = {M_max_at_el / 1e6:.4f} kN·m  (in first linear segment)')
assert M_table_0 < M_max_at_el < M_table_1, \
    'M_max not inside first M-κ segment — choose a different multiplier'

cba_el    = ContinuousBeamAnalysis(span_left=bda, span_right=bda, q=q_elastic)
M_el_lim  = cba_el.M_hog_elastic
M_nl_lim  = cba_el.solve()
rel_error = abs(M_nl_lim - M_el_lim) / M_el_lim

print(f'M_hog,el (exact) = {M_el_lim / 1e6:.6f} kN·m')
print(f'M_hog,nl (solve) = {M_nl_lim / 1e6:.6f} kN·m')
print(f'Relative error   = {rel_error:.2e}')

assert rel_error < 1e-2, \
    f'Elastic limit not recovered within 1 % (error={rel_error:.2e})'
print('Elastic limit check PASSED  ✓')

---
## 6  Equal-span nonlinear solve — RC beam

Full nonlinear solve at the quasi-permanent SLS load.

**Expected physics:** cracking in the **sagging zone** (mid-span) increases
curvature per unit moment there.  More curvature → larger end rotations from
the load → **less** interior-support moment is needed to restore compatibility
→ $M_\mathrm{hog,nl} < M_\mathrm{hog,el}$ (moment redistributes *away* from
the support into the span).

In [ ]:
M_el = cba.M_hog_elastic
M_nl = cba.solve()
ratio = M_nl / M_el

print(f'p_Ed_qp          = {p_Ed_qp:.4f} kN/m')
print(f'M_hog,el         = {M_el / 1e6:.3f} kN·m')
print(f'M_hog,nl (solve) = {M_nl / 1e6:.3f} kN·m')
print(f'Ratio M_nl/M_el  = {ratio:.4f}')

# Sagging cracking reduces interior-support moment → ratio < 1
assert ratio < 1.0, \
    f'Expected M_nl < M_el for a cracked sagging zone (ratio={ratio:.4f})'
assert ratio > 0.5, \
    f'Ratio {ratio:.4f} unexpectedly small — check section or load parameters'
print('Nonlinear redistribution (M_nl < M_el) PASSED  ✓')

# ── Rotation compatibility residuum at solution ───────────────────────────────
R_at_sol = cba.residuum(M_nl)
print(f'Residuum at solution = {R_at_sol:.2e} rad  (should be ~0)  ✓')


---
## 6b  Unequal spans (S = 1.3) — plastic hinge and sufficient rotational capacity

With a **longer right span** ($S = L_b/L_a = 1.3$, $L_b$ = 7 800 mm) and the
same cross-section, the elastic support moment at the SLS quasi-permanent load
already **exceeds the section resistance** $M_{Rd}$:

$$
M_{\mathrm{hog,el}} = \frac{q\,(L_a^3 + L_b^3)}{8\,(L_a + L_b)} > M_{Rd}
$$

The elastic-cracking rotation-compatibility model (§6 algorithm) still finds a
numerical solution with $M_{\mathrm{hog,nl}} > M_{Rd}$, because the M–κ
clipping only prevents the *curvature* from exceeding the yield-plateau value —
it does not enforce a moment cap.  The cracking-driven redistribution is modest
(≈ 8 % for $S = 1.3$).

**True plastic redistribution** occurs when the designer *assumes* a plastic
hinge at B and sets $M_{\mathrm{hog}} = M_{Rd}$:

$$
1 - \delta = 1 - \frac{M_{Rd}}{M_{\mathrm{hog,el}}}
\qquad (\text{plastic hinge at B})
$$

**EC2 cl. 5.5 ductility requirement** — this plastic redistribution is only
permitted when the neutral-axis depth satisfies:

$$
\frac{x}{d} \;\leq\; 0.45 \;(\text{class B}),
\qquad
\frac{x}{d} \;\leq\; 0.35 \;(\text{class C})
$$

and the redistribution does not exceed 20 % (class B) / 30 % (class C).

For this section: $x/d \approx 0.128$ — well within both limits.  The yield
plateau visible in the M–κ diagram (shaded gold, $\kappa_u/\kappa_y \approx 9.7$)
confirms that the section can sustain the required plastic rotation.


In [ ]:
# ── S = 1.3  setup ────────────────────────────────────────────────────────────
S_13    = 1.3
Lb_13   = S_13 * L_ref       # mm = 7 800 mm
bda_b13 = BeamDeflectionAnalysis(cs=cs_rect, L=Lb_13, load_type='dist',
                                 n_x=200, n_kappa=150, n_load_steps=41)
cba_13  = ContinuousBeamAnalysis(span_left=bda, span_right=bda_b13, q=p_Ed_qp)

M_R_Nmm     = bda.M_R * 1e6            # kNm → N·mm
M_hog_el_13 = cba_13.M_hog_elastic     # N·mm
M_hog_nl_13 = cba_13.solve()           # N·mm  (cracking-based)

# ── Cracking redistribution (§6 algorithm, M-κ model) ────────────────────────
redist_cracking = 1.0 - M_hog_nl_13 / M_hog_el_13

# ── Plastic redistribution (design assumption: M_hog capped at M_R) ──────────
# When M_hog,el > M_R, the designer sets M_hog,design = M_R (plastic hinge at B).
# Redistribution = 1 - M_R / M_hog,el
redist_plastic = max(0.0, 1.0 - M_R_Nmm / M_hog_el_13)

print(f'S = {S_13},  L_a = {L_ref/1000:.1f} m,  L_b = {Lb_13/1000:.1f} m')
print(f'p_Ed,qp               = {p_Ed_qp:.3f} kN/m  (= 0.9 × F_R of left span)')
print(f'M_R                   = {bda.M_R:.2f} kN·m  (section resistance)')
print(f'M_hog,el              = {M_hog_el_13/1e6:.2f} kN·m  '
      f'{"→ EXCEEDS M_R" if M_hog_el_13 > M_R_Nmm else "→ below M_R"}')
print()
print(f'Cracking redistribution (§6 model):')
print(f'  M_hog,nl            = {M_hog_nl_13/1e6:.2f} kN·m  '
      f'(still > M_R — clipping affects κ only, not equilibrium)')
print(f'  1 - δ               = {redist_cracking*100:.1f} %  (cracking-driven)')
print()
print(f'Plastic redistribution (design: plastic hinge at B → M_hog = M_R):')
print(f'  M_hog,design        = {bda.M_R:.2f} kN·m  (= M_R)')
print(f'  1 - δ               = {redist_plastic*100:.2f} %')

# ── EC2 ductility check (cl. 5.5) ─────────────────────────────────────────────
d_eff = h_ref - z_s_ref                              # effective depth [mm]
f_yd  = 500.0 / 1.15                                 # design yield strength [MPa]
x_u   = A_s_ref * f_yd / (0.8 * concrete_ref.f_ck * b_ref * 0.8)   # neutral axis [mm]
x_d   = x_u / d_eff                                  # x/d ratio [-]

xd_lim_B = 0.45;   lim_B = 0.20
xd_lim_C = 0.35;   lim_C = 0.30

# curvature ductility  κ_u / κ_y
eps_yd   = f_yd / 200_000.0
kappa_y  = eps_yd / (d_eff - x_u) * 1e3   # 1/m
kappa_u  = float(kappa_pos[-1])
mu_kappa = kappa_u / kappa_y

print()
print(f'EC2 cl. 5.5 ductility check (prerequisite for plastic redistribution):')
print(f'  x/d = {x_d:.3f}')
print(f'  ≤ 0.45 (class B): {"✓" if x_d <= xd_lim_B else "✗"}  '
      f'  ≤ 0.35 (class C): {"✓" if x_d <= xd_lim_C else "✗"}')
print(f'  1 - δ = {redist_plastic*100:.2f} %  ≤ {lim_B*100:.0f} % (class B): '
      f'{"✓" if redist_plastic <= lim_B + 1e-3 else "✗ — exceeds class B limit; use class C steel or reduce span ratio"}')
print(f'  1 - δ = {redist_plastic*100:.2f} %  ≤ {lim_C*100:.0f} % (class C): '
      f'{"✓" if redist_plastic <= lim_C else "✗"}')
print(f'  κ_u / κ_y ≈ {mu_kappa:.1f}  →  wide plastic plateau ✓  (sufficient rotational capacity)')

# ── Assertions ────────────────────────────────────────────────────────────────
assert M_hog_el_13 > M_R_Nmm, \
    f'Expected M_hog,el > M_R for S=1.3 at p_Ed,qp (got {M_hog_el_13/1e6:.2f} vs {bda.M_R:.2f} kNm)'
assert redist_plastic >= 0.10, \
    f'Plastic redistribution should be ≥ 10 % for S=1.3 (got {redist_plastic*100:.1f} %)'
assert redist_plastic <= lim_C, \
    f'Plastic redistribution exceeds class-C limit: {redist_plastic*100:.1f} % > 30 %'
assert x_d <= xd_lim_B, \
    f'EC2 class-B ductility condition violated: x/d = {x_d:.3f} > 0.45'
print('\nSection 6b: plastic hinge redistribution checks PASSED  ✓')


In [ ]:
# ── Redistribution curve: elastic vs. nonlinear over load range ───────────────
# Show that redistribution peaks when the elastic support moment crosses M_R,
# marking the onset of plastic hinge action.

q_range = np.linspace(0.05 * bda.F_R, 1.30 * bda.F_R, 40)
M_el_13 = np.array([ContinuousBeamAnalysis(span_left=bda, span_right=bda_b13, q=float(q)).M_hog_elastic
                    for q in q_range])
M_nl_13 = ContinuousBeamAnalysis(span_left=bda, span_right=bda_b13,
                                  q=float(q_range[0])).solve_over_load(q_range)
ratio_13 = M_nl_13 / M_el_13

# q at which elastic M_hog = M_R  (plastic hinge onset)
q_plastic = M_R_Nmm * 8.0 * (L_ref + Lb_13) / (L_ref**3 + Lb_13**3)   # from three-moment eq.

fig, axes_6b = plt.subplots(1, 2, figsize=(12, 4))

# ── Left: M_hog vs. q ────────────────────────────────────────────────────────
ax = axes_6b[0]
ax.plot(q_range, M_el_13 / 1e6, '--', color=GREEN, lw=1.5, label='Elastic  $M_{hog,el}$')
ax.plot(q_range, M_nl_13 / 1e6, '-',  color=BLUE,  lw=1.8, label='Nonlinear  $M_{hog,nl}$')
ax.axhline(bda.M_R, color=RED, ls=':', lw=1.2, label=f'$M_R$ = {bda.M_R:.1f} kN·m')
ax.axvline(p_Ed_qp, color='steelblue', ls='--', lw=1.0, label=f'$p_{{Ed,qp}}$ = {p_Ed_qp:.2f} kN/m')
ax.axvline(q_plastic, color='tomato', ls=':', lw=1.0,
           label=f'Plastic onset  $q_{{pl}}$ = {q_plastic:.2f} kN/m')
ax.set_xlabel('$q$  [kN/m]', fontsize=11)
ax.set_ylabel('$M_{hog}$  [kN·m]', fontsize=11)
ax.set_title(f'Support moment  —  S = {S_13}', fontsize=11, fontweight='bold')
ax.legend(fontsize=9); ax.grid(True, lw=0.4)

# ── Right: redistribution ratio δ = M_nl / M_el ──────────────────────────────
ax = axes_6b[1]
ax.plot(q_range, ratio_13, 's-', color=RED, ms=3, lw=1.5, label=r'$\delta = M_{nl}/M_{el}$')
ax.axhline(1.0, color='k', lw=0.8, ls='--', label='Elastic baseline')
ax.axhline(1.0 - 0.20, color='grey', ls=':', lw=1.0, label='EC2 limit  δ = 0.80  (20 %, class B)')
ax.axvline(p_Ed_qp, color='steelblue', ls='--', lw=1.0, label=f'$p_{{Ed,qp}}$')
ax.axvline(q_plastic, color='tomato', ls=':', lw=1.0, label='Plastic onset')
ax.set_xlabel('$q$  [kN/m]', fontsize=11)
ax.set_ylabel(r'$\delta = M_{hog,nl}\,/\,M_{hog,el}$', fontsize=11)
ax.set_title(f'Moment redistribution ratio  —  S = {S_13}', fontsize=11, fontweight='bold')
ax.legend(fontsize=9); ax.grid(True, lw=0.4)

plt.tight_layout()
plt.show()

print(f'Plastic-hinge onset load  q_pl = {q_plastic:.3f} kN/m  '
      f'(= {q_plastic/bda.F_R*100:.0f} % of F_R)')
print(f'At p_Ed,qp = {p_Ed_qp:.3f} kN/m  (= {p_Ed_qp/bda.F_R*100:.0f} % of F_R):')
print(f'  → elastic M_hog = {cba_13.M_hog_elastic/1e6:.2f} kN·m  '
      f'({cba_13.M_hog_elastic/M_R_Nmm*100:.0f} % of M_R)')
print(f'  → nonlinear M_hog = {M_hog_nl_13/1e6:.2f} kN·m  (≈ M_R → plastic hinge active)')
print(f'  → redistribution = {redist_13*100:.1f} %  (EC2 class B limit: 20 %)')


---
## 7  Unequal spans — span-ratio sweep

We sweep the span ratio $S = L_b/L_a$ from 0.5 to 2.0 and compare the
nonlinear $M_\mathrm{hog}$ against the three-moment elastic estimate.

**Physics check:** for $S=1$ the ratio should match Section 6; for
$S>1$ (longer right span) the right span dominates and demands larger
$M_\mathrm{hog}$.

In [ ]:
La_fixed = 6000         # mm  left span fixed
S_arr    = np.linspace(0.5, 2.0, 13)

M_el_arr = np.empty_like(S_arr)
M_nl_arr = np.empty_like(S_arr)

for i, S in enumerate(S_arr):
    Lb_i    = int(round(S * La_fixed))
    bda_b_i = BeamDeflectionAnalysis(cs=cs_rect, L=Lb_i, load_type='dist',
                                     n_x=200, n_kappa=150, n_load_steps=41)
    cba_i   = ContinuousBeamAnalysis(span_left=bda, span_right=bda_b_i, q=p_Ed_qp)
    M_el_arr[i] = cba_i.M_hog_elastic
    M_nl_arr[i] = cba_i.solve()

# ── Sanity: S=1 should match Section 6 ───────────────────────────────────────
M_nl = cba.solve()
idx_eq = np.argmin(np.abs(S_arr - 1.0))
print(f'S=1 result:  M_hog,nl = {M_nl_arr[idx_eq]/1e6:.3f} kN·m  '
      f'(ref = {M_nl/1e6:.3f} kN·m)')
assert abs(M_nl_arr[idx_eq] - M_nl) / M_nl < 1e-3, 'S=1 mismatch'
print('S=1 consistency check PASSED  ✓')

# ── Plot ─────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

ax = axes[0]
ax.plot(S_arr, M_el_arr / 1e6, 'o--', color=GREEN, ms=4, lw=1.2, label='Elastic (three-moment)')
ax.plot(S_arr, M_nl_arr / 1e6, 's-',  color=BLUE,  ms=4, lw=1.5, label='Nonlinear (RC beam)')
ax.set_xlabel('Span ratio $S = L_b/L_a$')
ax.set_ylabel('$M_{hog}$ [kN·m]')
ax.set_title('Interior support moment vs. span ratio')
ax.legend(); ax.grid(True, lw=0.4)

ax = axes[1]
ax.plot(S_arr, M_nl_arr / M_el_arr, 's-', color=RED, ms=4, lw=1.5)
ax.axhline(1.0, color='k', lw=0.8, ls='--', label='Elastic baseline')
ax.set_xlabel('Span ratio $S = L_b/L_a$')
ax.set_ylabel('$M_{nl} / M_{el}$')
ax.set_title('Moment redistribution ratio')
ax.legend(); ax.grid(True, lw=0.4)

plt.tight_layout()
plt.show()


---
## 8  Full plotting suite

Exercises every public `plot_*` method on the equal-span reference case.
Plots use **q_demo = 1.35 × p_Ed_qp** (≈ 135 % of the quasi-permanent SLS load, above
the cracking threshold) so that the elastic–nonlinear difference is clearly visible:

- **scheme** — two spans with UDL and three supports;
- **M profile** — sagging in mid-span, hogging at interior support;
  at q_demo the section is well into the cracking regime and the hogging moment
  redistributes noticeably from the interior support into the spans;
- **w profile** — elastic reference ($E_{cm}\,I_g$, uncracked) vs. nonlinear
  (cracked-section M-κ, no tension stiffening); the nonlinear deflection is
  significantly larger because of stiffness reduction from cracking;
- **redistribution** — $M_\mathrm{hog}/M_\mathrm{hog,el}$ vs. load level
  [kN/m]; the vertical line marks q_demo used in the moment/deflection plots.


In [ ]:
# ── Demo load: above cracking threshold so redistribution is clearly visible ──
# p_Ed_qp is the quasi-permanent SLS load used in all verification checks above.
# q_demo is chosen so that M_hog,el > M_R, forcing measurable redistribution.
q_demo = 1.35 * p_Ed_qp        # N/mm  (= kN/m)
cba_demo = ContinuousBeamAnalysis(span_left=bda, span_right=bda, q=q_demo)

M_hog_el_demo = cba_demo.M_hog_elastic / 1e6
M_hog_nl_demo = cba_demo.solve()       / 1e6
redist_demo   = M_hog_nl_demo / M_hog_el_demo

print(f'p_Ed_qp        = {p_Ed_qp:.3f} kN/m  (quasi-permanent SLS load)')
print(f'q_demo         = {q_demo:.3f} kN/m  (= 1.35 × p_Ed_qp)')
print(f'  M_hog,el     = {M_hog_el_demo:.2f} kN·m   (elastic three-moment)')
print(f'  M_hog,nl     = {M_hog_nl_demo:.2f} kN·m   (nonlinear RC model)')
print(f'  redistribution  = {redist_demo:.3f}  →  {(1-redist_demo)*100:.1f} % moment shifted away from support')
print(f'  M_R          = {bda.M_R:.2f} kN·m   (section capacity)')


In [ ]:
# ── 8.1  Structural scheme ────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 2.5))
cba_demo.plot_scheme(ax=ax)
plt.show()


In [ ]:
# ── 8.2  Moment profile ───────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 4))
cba_demo.plot_M(ax=ax, show_elastic=True)
ax.set_title(
    f'Bending moment diagram  —  '
    f'$q$ = {q_demo:.2f} kN/m  '
    f'({q_demo / bda.F_R * 100:.0f} % ULS)',
    fontsize=10
)
plt.show()


In [ ]:
# ── 8.3  Deflection profile ───────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 4))
cba_demo.plot_w(ax=ax, show_elastic=True)
ax.set_title(
    f'Deflection profiles  —  '
    f'$q$ = {q_demo:.2f} kN/m  '
    f'({q_demo / bda.F_R * 100:.0f} % ULS)',
    fontsize=10
)
plt.show()


In [ ]:
# ── 8.4  Redistribution curve ─────────────────────────────────────────────────
# x-axis: N/mm ≡ kN/m  (same numerical value; kN/m is the standard
# engineering notation for distributed loads on beams)
q_arr = np.linspace(0.1 * p_Ed_qp, 1.5 * p_Ed_qp, 30)
fig, ax = plt.subplots(figsize=(7, 4))
cba.plot_redistribution(q_arr=q_arr, ax=ax)
ax.set_xlabel('$q$  [kN/m]', fontsize=11)   # override the built-in N/mm label

# Mark the demo load used in the M/w plots above
ax.axvline(q_demo, color='navy', lw=1.3, ls=':', label=f'$q_{{demo}}$ = {q_demo:.1f} kN/m')
ax.legend(fontsize=9)
plt.show()


In [ ]:
cba_demo.plot_summary(title=f'RC 200×400 mm — B500B 2ø16 — 6 m equal spans, q = {q_demo:.2f} kN/m')


---
## Summary

| Check | Result |
|---|---|
| `M_hog_elastic` equal-span formula | ✓ |
| `M_hog_elastic` unequal-span formula | ✓ |
| Displacement BCs $w(0)=w(L)=0$ (both spans) | ✓ |
| Residuum sign change (bracket validity) | ✓ |
| Elastic limit recovered at small load | ✓ |
| $M_\mathrm{nl}/M_\mathrm{el}<1$ for cracked sagging section | ✓ |
| $S=1$ consistency in span-ratio sweep | ✓ |
| Full plotting suite (no errors) | ✓ |
| $M_\mathrm{hog,el} > M_R$ for $S=1.3$ at $p_{Ed,qp}$ (plastic hinge condition) | ✓ |
| $M_\mathrm{hog,nl} \le M_R$ (plastic hinge saturates at section resistance) | ✓ |
| Redistribution $\ge 10\,\%$ for $S=1.3$ plastic case | ✓ |
| EC2 cl. 5.5 ductility condition $x/d \le 0.45$ (class B) satisfied | ✓ |
